In [5]:
# Install CPU-only torch ecosystem (CUDA 13 incompatible with driver 470/CUDA 11.4)
%pip install --force-reinstall --no-deps --index-url https://download.pytorch.org/whl/cpu torch==2.14.0 torchaudio torchvision --user
%pip install --upgrade transformers --user

Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached torch-2.14.0%2Bcpu-cp311-cp311-manylinux_2_28_aarch64.whl.metadata (37 kB)
  Using cached torchaudio-2.11.0%2Bcpu-cp311-cp311-manylinux_2_28_aarch64.whl.metadata (6.9 kB)
  Using cached torchvision-0.29.0%2Bcpu-cp311-cp311-manylinux_2_28_aarch64.whl.metadata (5.7 kB)
Using cached torch-2.14.0%2Bcpu-cp311-cp311-manylinux_2_28_aarch64.whl (159.2 MB)
Using cached torchaudio-2.11.0%2Bcpu-cp311-cp311-manylinux_2_28_aarch64.whl (335 kB)
Using cached torchvision-0.29.0%2Bcpu-cp311-cp311-manylinux_2_28_aarch64.whl (2.1 MB)
  Attempting uninstall: torchaudio
    Found existing installation: torchaudio 2.11.0+cpu
    Uninstalling torchaudio-2.11.0+cpu:
      Successfully uninstalled torchaudio-2.11.0+cpu
  Attempting uninstall: torchvision━━━━━━━━━━━━━ 0/3 [torchaudio]
    Found existing installation: torchvision 0.29.0+cpu/3 [torchaudio]
    Uninstalling torchvision-0.29.0+cpu:━━━━ 0/3 [torchaudio]
      Successfully uninst

In [6]:
# import AutoTokenizer
from transformers import AutoTokenizer

In [7]:
# Tokenization is the process of converting a text document or sentence to smaller units.
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
input_text = "What is unhappiness?"
tokens = tokenizer.tokenize(input_text, return_tensors="pt")

# This is a example of subword tokenazation
print(f"{tokens = }")

tokens = ['what', 'is', 'un', '##ha', '##pp', '##iness', '?']


In [9]:
# Token embeddings, which convert tokens to numerical vectors.
from transformers import BertTokenizer, BertModel
import torch

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

input_text='''
After a long day at work, Sarah decided to relax by taking her
dog for a walk in the park. As they strolled along the
tree-lined paths, Sarah's dog, Max, eagerly sniffed around,
chasing after squirrels and birds. Sarah smiled as she watched
Max enjoy himself, feeling grateful for the companionship and
joy that her furry friend brought into her life.
'''

tokens = tokenizer(input_text, return_tensors="pt")

with torch.no_grad():
    # This is for generating token embeddings
    outputs = model(**tokens)

# Extract token embeddings
last_hidden_states = outputs.last_hidden_state

print("Token embeddings: ")

# print all the token embeddings for each word
for token, embedding in zip(tokens["input_ids"][0], last_hidden_states[0]):
    word = tokenizer.decode(int(token))
    print(f"{word}: {embedding}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9964.98it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'input_ids': tensor([[  101,  2044,  1037,  2146,  2154,  2012,  2147,  1010,  4532,  2787,
          2000,  9483,  2011,  2635,  2014,  3899,  2005,  1037,  3328,  1999,
          1996,  2380,  1012,  2004,  2027, 20354,  2247,  1996,  3392,  1011,
          7732, 10425,  1010,  4532,  1005,  1055,  3899,  1010,  4098,  1010,
         17858, 18013,  2105,  1010, 11777,  2044, 29384,  1998,  5055,  1012,
          4532,  3281,  2004,  2016,  3427,  4098,  5959,  2370,  1010,  3110,
          8794,  2005,  1996, 11946,  5605,  1998,  6569,  2008,  2014, 28662,
          2767,  2716,  2046,  2014,  2166,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [ ]:
# Convert high-dimensional into a 2D space
# install scikit-learn
%pip install scikit-learn
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Reducing demensionality using t-SNE with lower perplexity
tsne = TSNE(n_components=2, perplexity=5, random_state=42)
embeddings_tsne = tsne.fit_transform(last_hidden_states[0])

# Plot the embeddings on a 2D graph
plt.figure(figsize=(10, 8))
plt.scatter(embeddings_tsne[:, 0],
            embeddings_tsne[:, 1], marker='o')
for i, word in enumerate(tokenizer.convert_ids_to_tokens(
    tokens["input_ids"][0])):
    plt.annotate(word, xy=(embeddings_tsne[i, 0],
                          embeddings_tsne[i, 1]), 
                          fontsize=10)
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.title('t-SNE Visalization of Token Embeddings')
plt.show()

'''
Sumary in here word embeddings allow you to see which words are often used together. 
This technique captures semantic relationships between words based on their usage patterns in large text corpora
'''

In [ ]:
# Positional encoding plays a crucial role in transformer-based models by imparting essential positional information about the order of tokens within a sequence. 
# Access the embeddings layer directly
embeddings = model.embeddings
# Extract the positional embeddings
positional_embeddings = embeddings.position_embeddings.weight
# Gets the positional encodings for the input tokens
position_ids = torch.arange(tokens['input_ids'].size(1),
                            dtype=torch.long).unsqueeze(0)
# Extract the position IDs from the input tokens
input_positional_embeddings = positional_embeddings[position_ids]

print("Positional embeddings shape:", input_positional_embeddings.shape)
print("Positional embeddings for each token:")

for token_id, pos_embedding in zip(tokens['input_ids'][0], input_positional_embeddings[0]):
    token = tokenizer.decode([token_id])
    print(f"{token}: {pos_embedding}")

'''
Summary here postional encoding is like give to know tokens order before process to transformer
'''

In [ ]:
# Transformer for do Sentiment analysis

In [ ]:
# Sentiment analysis is an NLP technique that determines the sentiment expressed in a piece of text. 
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased-finetuned-sst-2-english")

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased-finetuned-sst-2-english")

In [ ]:
import torch

text = "I loved the movie, it was fantastic!"

# Tokenizes the text
inputs = tokenizer(text, return_tensors = "pt")
print(inputs)

In [ ]:
# Performs inference
outputs = model(**inputs)
print(outputs)

'''
Summary here logits [[-4.3428, 4.6955]]; it is a mean model 
It gives 1 or positive because the second element has a higher value
'''

In [ ]:
# Gets the predict label (0 for negative and 1 for positive)
predicted_label = torch.argmax(outputs.logits)
sentiment = "positive" if predicted_label == 1 else "negative"

print("Predicted sentiment:", sentiment)

In [ ]:
# Own Experiment for negative sentiment
bad_review = "I hate this movie and it is so boring"

# Tokenizing process
input = tokenizer(text, return_tensors = "pt")

# Inferencing process
outputs = model(**inputs)
print(outputs)

'''
Summary here logits [[ 4.3705, -3.5111]]; it will give 0 or negative because 
the first element has a higher value
'''

In [ ]:
# Transformers Pipeline

In [45]:
from transformers import pipeline

# Showing a list of supported tasks
try:
    dummy_pipeline = pipeline(task="dummy")
except Exception as e:
    print(e)

"Unknown task dummy, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"


In [ ]:
# Use text-classification
classifier = pipeline(
    "text-classification", 
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

review1 = '''From the warm welcome to the exquisite dishes and impeccable
service, dining at Gourmet Haven is an unforgettable experience that
leaves you eager to return.'''

review2 = '''Despite high expectations, our experience at Savor Bistro
fell short; the food was bland, service was slow, and the overall
atmosphere lacked charm, leaving us disappointed and unlikely to
revisit.'''

print(classifier([review1, review2]))

In [ ]:
# Use text-generation
from transformers import pipeline

generator = pipeline("text-generation", model="openai-community/gpt2")

generator("In this course, we will teach you how to")

In [ ]:
# controling output using max_length and num_return_sequences
generator(
            "Inthis course, we will teach you how to",
            max_length = 50,
            num_return_sequences = 3
         )

In [ ]:
# Use Text Summaryzation
from transformers import pipeline

# This is old way
# summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# This is modern way
summarizer = pipeline(task="text-generation", model="facebook/bart-large-cnn")

article = """
A quantum computer is a computer that exploits quantum mechanical
phenomena. At small scales, physical matter exhibits properties of
both particles and waves, and quantum computing leverages this
behavior using specialized hardware. Classical physics cannot
explain the operation of these quantum devices, and a scalable
quantum computer could perform some calculations exponentially
faster than any modern "classical" computer. In particular, a
large-scale quantum computer could break widely used encryption
schemes and aid physicists in performing physical simulations;
however, the current state of the art is still largely
experimental and impractical.
The basic unit of information in quantum computing is the qubit,
similar to the bit in traditional digital electronics. Unlike a
classical bit, a qubit can exist in a superposition of its two
"basis" states, which loosely means that it is in both states
simultaneously. When measuring a qubit, the result is a
probabilistic output of a classical bit. If a quantum computer
manipulates the qubit in a particular way, wave interference
effects can amplify the desired measurement results. The design
of quantum algorithms involves creating procedures that allow a
quantum computer to perform calculations efficiently.
Physically engineering high-quality qubits has proven challenging.
If a physical qubit is not sufficiently isolated from its
environment, it suffers from quantum decoherence, introducing noise
into calculations. National governments have invested heavily in
experimental research that aims to develop scalable qubits with
longer coherence times and lower error rates. Two of the most
promising technologies are superconductors (which isolate an
electrical current by eliminating electrical resistance) and ion
traps (which confine a single atomic particle using electromagnetic
fields).
Any computational problem that can be solved by a classical computer
can also be solved by a quantum computer.[2] Conversely, any problem
that can be solved by a quantum computer can also be solved by a
classical computer, at least in principle given enough time. In other
words, quantum computers obey the Church–Turing thesis. This means
that while quantum computers provide no additional advantages over
classical computers in terms of computability, quantum algorithms
for certain problems have significantly lower time complexities than
corresponding known classical algorithms. Notably, quantum computers
are believed to be able to solve certain problems quickly that no
classical computer could solve in any feasible amount of time—a feat
known as "quantum supremacy." The study of the computational
complexity of problems with respect to quantum computers is known as
quantum complexity theory.
"""

prompt = f"Summarize the following text:\n\n{article}"

result = summarizer(prompt, min_length=100, max_length=250, do_sample=False)
print(result[0]['generated_text'])

In [ ]:
# Try with do_sample = True
print(summarizer(article, min_length = 100, max_length = 250, do_sample = True))
print(result[0]['generated_text'])

In [ ]:
# Use Text Translation
from transformers import pipeline

# This is old way
# translator = pipeline("translation", model = "google-t5/t5-base")

# This is modern way
translator = pipeline("text-generation", model="google-t5/t5-base")

prompt = "translate English to German: How are you?"

result = translator(prompt)
print(result[0]['generated_text'])

'''
Summary this is not success becasue transformers is new version
'''

In [30]:
# Zero-shot Classification is Zero-shot classification is a machine learning technique where a model can correctly classify new items or classes that it has never seen and was never explicitly trained on.
!pip install sentencepiece
!pip install protobuf
!pip install tiktoken

In [31]:
# Use zer-shot-classification
from transformers import pipeline
zero_shot_classifier = pipeline("zero-shot-classification", 
                                model='joeddav/xlm-roberta-large-xnli')

text1 = '''
"In the intricate realm of global affairs, the interplay of power,
diplomacy, and governance stands as a defining force in the
trajectory of nations. Amidst fervent debates in legislative
chambers and pivotal dialogues among world leaders, ideologies
clash and policies take shape, shaping the course of societies.
Issues such as economic disparity, environmental stewardship, and
human rights take precedence, driving conversations and shaping
public sentiment. In an age of digital interconnectedness, social
media platforms have emerged as influential channels for discourse
and activism, amplifying voices and reshaping narratives with
remarkable speed and breadth. As citizens grapple with the
complexities of contemporary governance, the pursuit of accountable
and transparent leadership remains paramount, reflecting an
enduring quest for fairness and inclusivity in societal governance."
'''
text2 = '''
In the tender tapestry of human connection, romance weaves its
delicate threads, binding hearts in a dance of passion and longing.
From the flutter of a first glance to the warmth of an intimate
embrace, love blooms in the most unexpected places, transcending
barriers of time and circumstance. In the gentle caress of a hand
and the whispered promises of affection, two souls find solace in
each other's embrace, navigating the complexities of intimacy with
tender care. As the sun sets and stars illuminate the night sky,
lovers share stolen moments of intimacy, lost in the intoxicating
rhythm of each other's presence. In the symphony of love, every
glance, every touch, speaks volumes of a shared bond that defies
explanation, leaving hearts entwined in an eternal embrace.
'''

'''
Sumarry here:
- Zero-shot classification is a machine learning technique where a model can correctly classify new items or classes that it has never seen and was never explicitly trained on. Instead of relying on specific training examples for that class, it uses its prior general knowledge (such as understanding semantic concepts, descriptions, or visual attributes) to reason and make a prediction.
- One-shot classification is a technique where a model is trained to recognize a new class using only a single training example. The goal is to teach the model to generalize and recognize similar objects in the future based entirely on that one sample.
'''

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,labels,scores
0,politics,0.990988
1,technology,0.828363
2,romance,0.465752
3,business,0.278939


'\nSumarry here:\n- Zero-shot classification is a machine learning technique where a model can correctly classify new items or classes that it has never seen and was never explicitly trained on. Instead of relying on specific training examples for that class, it uses its prior general knowledge (such as understanding semantic concepts, descriptions, or visual attributes) to reason and make a prediction.\n- One-shot classification is a technique where a model is trained to recognize a new class using only a single training example. The goal is to teach the model to generalize and recognize similar objects in the future based entirely on that one sample.\n'

In [33]:
# Convert to pandas for easy to view
import pandas as pd

candidate_labels = ["technology", "politics", "business", "romance"]
prediction = zero_shot_classifier(text1, candidate_labels, multi_label = True)

display(pd.DataFrame(prediction).drop(["sequence"], axis=1))

,labels,scores
0,politics,0.990988
1,technology,0.828363
2,romance,0.465752
3,business,0.278939


In [34]:
prediction = zero_shot_classifier([text1, text2], candidate_labels, multi_label = True)

display(pd.DataFrame(prediction).drop(["sequence"], axis=1))

,labels,scores
0,"[politics, technology, romance, business]","[0.9909884929656982, 0.828362762928009, 0.4657..."
1,"[romance, business, politics, technology]","[0.9982308149337769, 0.1188003197312355, 0.016..."


In [36]:
# zero-shot for image classification
from transformers import pipeline

classifier = pipeline("zero-shot-image-classification", 
                      model = "openai/clip-vit-large-patch14-336")

labels_for_classification = ["airplane", "car", "train"]
scores = classifier("cessna_1726.jpg",
candidate_labels = labels_for_classification)
pd.DataFrame(scores)

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

,score,label
0,0.998503,airplane
1,0.001475,car
2,0.000022,train


In [44]:
# Use Question-answering tasks
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

# old way
# QA_model = pipeline(task = "question-answering", model = "deepset/roberta-base-squad2")

# modern way
model_name = "deepset/roberta-base-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

text = '''
The English name of "Singapore" is an anglicisation of the native
Malay name for the country, Singapura (pronounced [siŋapura]),
which was in turn derived from the Sanskrit word for 'lion city'
(Sanskrit: सिंहपुर; romanised: Siṃhapura; Brahmi: 𑀲𑀺𑀁𑀳𑀧𑀼𑀭 ; literally
"lion city"; siṃha means 'lion', pura means 'city' or 'fortress' ).
Pulau Ujong was one of the earliest references to Singapore Island,
which corresponds to a Chinese account from the third century
referred to a place as Pú Luó Zhōng (Chinese: 蒲 羅 中), a
transcription of the Malay name for 'island at the end of a
peninsula'. Early references to the name Temasek (or Tumasik) are
found in the Nagarakretagama, a Javanese eulogy written in 1365,
and a Vietnamese source from the same time period. The name possibly
means Sea Town, being derived from the Malay tasek, meaning 'sea' or
'lake'. The Chinese traveller Wang Dayuan visited a place around 1330
named Danmaxi (Chinese: 淡馬錫; pinyin: Dànmǎxí; Wade–Giles: Tan Ma Hsi)
or Tam ma siak, depending on pronunciation; this may be a transcription
of Temasek, alternatively, it may be a combination of the Malay Tanah
meaning 'land' and Chinese xi meaning 'tin', which was traded on the
island
'''

question = "What is the meaning of Singapura?"

# 2. Proses input teks dan pertanyaan menjadi token tensor
inputs = tokenizer(question, text, return_tensors="pt")

# 3. Jalankan model untuk memprediksi posisi awal dan akhir jawaban
with torch.no_grad():
    outputs = model(**inputs)

# 4. Ambil indeks token dengan nilai probabilitas tertinggi
answer_start = torch.argmax(outputs.start_logits)
answer_end = torch.argmax(outputs.end_logits) + 1

# 5. Decode token kembali menjadi teks jawaban yang murni
answer = tokenizer.decode(inputs.input_ids[0][answer_start:answer_end])

# 6. Susun ke dalam DataFrame agar formatnya mirip dengan keluaran pipeline lama
model_response = {
    "score": torch.max(torch.softmax(outputs.start_logits, dim=-1)).item(),
    "start": answer_start.item(),
    "end": answer_end.item(),
    "answer": answer.strip()
}

df = pd.DataFrame([model_response])
print(df)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      score  start  end     answer
0  0.269163     62   65  lion city
